# B1：企業客服 Agent — 完整架構決策

> **場景：** 台灣中型電商（年 GMV 約 30 億），每日約 5,000 個客服對話，現有 30 位真人客服。  
> **目標：** 70% 查詢由 AI 自動處理，剩下 30% 升級人工。

## 這本 Notebook 要回答的核心問題

每個架構決策都會問：**「為什麼用 X，不用 Y？」**

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 如何處理多種問題類型 | Intent Router | 讓單一 LLM 做所有事 |
| 知識庫如何搜尋 | Hybrid Search | 純向量搜尋 |
| 多輪對話狀態 | Summary Buffer | 全歷史塞 Context |
| 何時升級人工 | Rule + LLM 雙層判斷 | 純 LLM 判斷 |
| 向量資料庫 | Vertex AI Vector Search | Pinecone |
| 模型選擇 | Gemini Flash（FAQ）+ Pro（複雜） | 全用 Pro |

In [ ]:
import asyncio
import time
import json
import re
from typing import Literal
from dataclasses import dataclass, field
from enum import Enum

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm_fast = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # 模擬 Gemini Flash
    llm_smart = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 模擬 Gemini Pro
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

print("\n場景：台灣中型電商客服 Agent")
print("每日對話：5,000 | 目標 AI 處理率：70%")

## 系統架構全貌

```
用戶輸入
    │
    ▼
┌──────────────────────────────────────────────┐
│  Layer 1：Input Guard                        │
│  語言偵測 + PII 遮罩 + 惡意輸入過濾          │
└─────────────────────┬────────────────────────┘
                      │
                      ▼
┌──────────────────────────────────────────────┐
│  Layer 2：Intent Router（用 Gemini Flash）   │
│  訂單查詢 / 退換貨 / 技術支援 / 投訴 / 閒聊  │
└──────┬──────────┬──────────┬─────────────────┘
       │          │          │
       ▼          ▼          ▼
   訂單Agent   退換貨Agent  技術Agent
   (工具呼叫)  (RAG+工具)   (RAG)
       │          │          │
       └──────────┴──────────┘
                  │
                  ▼
┌──────────────────────────────────────────────┐
│  Layer 3：Escalation Judge                   │
│  決定：AI 回覆 or 升級人工                   │
└─────────────────────┬────────────────────────┘
                      │
              ┌───────┴───────┐
              ▼               ▼
          AI 回覆          升級人工
                         (含完整 Context)
```

## 決策 1：Intent Router — 為什麼不讓單一 LLM 做所有事？

**❌ 被拒絕的方案：One LLM to rule them all**  
讓一個 Gemini Pro 接收所有查詢，自己決定要查訂單、處理退換貨、或解答技術問題。

**問題：**
- System Prompt 越來越長（涵蓋所有場景的規則），導致 LLM 混淆
- 無法針對不同場景優化（訂單查詢只需要 tool，不需要 RAG）
- 成本：每個簡單的「訂單在哪裡」都呼叫貴的 Pro 模型
- 無法獨立評估和改進各場景的品質

**✅ 選擇的方案：Intent Router + Specialized Agents**

In [ ]:
class Intent(Enum):
    ORDER_QUERY   = "order_query"    # 查訂單狀態
    RETURN_REFUND = "return_refund"  # 退換貨
    TECH_SUPPORT  = "tech_support"   # 技術問題
    COMPLAINT     = "complaint"      # 投訴
    GENERAL_FAQ   = "general_faq"    # 一般問題
    CHITCHAT      = "chitchat"       # 閒聊

# 各 Intent 的路由策略
INTENT_CONFIG = {
    Intent.ORDER_QUERY: {
        "model": "flash",          # ✅ 不需要推理，工具查詢就夠了
        "requires_rag": False,
        "requires_tools": True,
        "tools": ["query_order_db", "get_tracking"],
        "avg_cost": "$0.001",
    },
    Intent.RETURN_REFUND: {
        "model": "pro",            # ✅ 需要理解政策 + 同理心
        "requires_rag": True,       # 退貨政策在知識庫
        "requires_tools": True,
        "tools": ["query_order_db", "create_return_ticket"],
        "avg_cost": "$0.008",
    },
    Intent.TECH_SUPPORT: {
        "model": "pro",
        "requires_rag": True,       # 技術文件在知識庫
        "requires_tools": False,
        "tools": [],
        "avg_cost": "$0.006",
    },
    Intent.COMPLAINT: {
        "model": "pro",
        "requires_rag": False,
        "requires_tools": False,
        "escalate_immediately": True,  # 投訴直接升人工
        "avg_cost": "$0.002",
    },
    Intent.GENERAL_FAQ: {
        "model": "flash",
        "requires_rag": True,
        "requires_tools": False,
        "tools": [],
        "avg_cost": "$0.003",
    },
    Intent.CHITCHAT: {
        "model": "flash",
        "requires_rag": False,
        "requires_tools": False,
        "tools": [],
        "avg_cost": "$0.0005",
    },
}

async def classify_intent(user_input: str) -> Intent:
    """用 Flash 做 Intent 分類（快、便宜）"""
    prompt = f"""將以下客服查詢分類為以下其中一個 intent：
order_query（查訂單）, return_refund（退換貨）, tech_support（技術問題）,
complaint（投訴）, general_faq（一般問題）, chitchat（閒聊）

查詢：{user_input}

只回覆 intent 名稱，不要其他文字。"""
    
    if LLM_AVAILABLE:
        response = await llm_fast.ainvoke([HumanMessage(content=prompt)])
        intent_str = response.content.strip().lower()
    else:
        # 模擬規則
        if "訂單" in user_input or "查詢" in user_input:
            intent_str = "order_query"
        elif "退" in user_input or "換" in user_input:
            intent_str = "return_refund"
        elif "投訴" in user_input or "抱怨" in user_input:
            intent_str = "complaint"
        else:
            intent_str = "general_faq"
    
    for intent in Intent:
        if intent.value == intent_str:
            return intent
    return Intent.GENERAL_FAQ

# 測試
test_queries = [
    "我的訂單 #1234 到哪了？",
    "這個商品壞掉了，我要退貨",
    "APP 一直閃退是什麼問題？",
    "你們的服務太差了！我要投訴！",
    "今天幾度？",
]

print("Intent 分類 + 路由策略：")
print("=" * 60)
for q in test_queries:
    intent = await classify_intent(q)
    config = INTENT_CONFIG[intent]
    model_icon = "⚡" if config["model"] == "flash" else "🧠"
    print(f"Q: {q}")
    print(f"   → {intent.value} | {model_icon} {config['model']} | RAG={config['requires_rag']} | Tools={config.get('tools', [])} | ~{config['avg_cost']}")
    print()

# 成本分析
daily_volume = {Intent.ORDER_QUERY: 2000, Intent.RETURN_REFUND: 800,
                Intent.TECH_SUPPORT: 600, Intent.GENERAL_FAQ: 1200,
                Intent.COMPLAINT: 200, Intent.CHITCHAT: 200}

print("月成本估算：")
total_daily = 0
for intent, volume in daily_volume.items():
    cost_per = float(INTENT_CONFIG[intent]["avg_cost"].replace("$", ""))
    daily_cost = volume * cost_per
    total_daily += daily_cost
    print(f"  {intent.value:<20} {volume:>5} 次/日 × {INTENT_CONFIG[intent]['avg_cost']:>8} = ${daily_cost:>6.2f}/日")

print(f"\n  月成本（30天）: ${total_daily * 30:,.0f}")
print(f"  如果全用 Pro（$0.008）: ${5000 * 0.008 * 30:,.0f}")
print(f"  成本節省: {(1 - total_daily * 30 / (5000 * 0.008 * 30)) * 100:.0f}%")

## 決策 2：混合搜尋 vs 純向量搜尋

**❌ 被拒絕：純向量搜尋**  
- 用戶輸入「訂單 #1234」→ 向量找不到精確的訂單號碼（語意距離無法做精確匹配）
- 「退貨政策第 3 條」→ 向量搜尋找不到「第 3 條」這種結構性查詢

**✅ 選擇：BM25 + 向量（Hybrid）**

In [ ]:
import math

# 模擬知識庫文件
KB_DOCUMENTS = [
    {"id": "faq_001", "content": "退貨政策：商品購買後 7 天內可申請退貨，需保持原包裝。",
     "keywords": ["退貨", "政策", "7天", "包裝"]},
    {"id": "faq_002", "content": "運費說明：訂單滿 500 元免運費，未達 500 元運費 60 元。",
     "keywords": ["運費", "500元", "免運", "60元"]},
    {"id": "faq_003", "content": "換貨流程：需在 14 天內申請，商品需為全新未使用狀態。",
     "keywords": ["換貨", "14天", "全新", "未使用"]},
    {"id": "faq_004", "content": "付款方式：支援信用卡、LINE Pay、超商繳費。",
     "keywords": ["付款", "信用卡", "LINE Pay", "超商"]},
    {"id": "faq_005", "content": "會員點數：每消費 100 元累積 1 點，1 點等於 1 元折扣。",
     "keywords": ["會員", "點數", "100元", "折扣"]},
]

def bm25_score(query: str, doc_keywords: list[str], k1=1.5, b=0.75) -> float:
    """簡化 BM25：精確關鍵字匹配（適合訂單號碼、規格數字）"""
    query_terms = set(query.lower().split())
    doc_terms = set(w.lower() for w in doc_keywords)
    matches = query_terms & doc_terms
    return len(matches) / (len(query_terms) + 0.01)

def vector_score(query: str, doc_content: str) -> float:
    """模擬向量相似度（Jaccard 近似）"""
    q = set(query.lower().split())
    d = set(doc_content.lower().split())
    if not (q | d): return 0
    return len(q & d) / len(q | d)

def hybrid_search(query: str, docs: list[dict], alpha: float = 0.5, top_k: int = 2) -> list[dict]:
    """
    混合搜尋：alpha × BM25 + (1-alpha) × 向量
    alpha=0.5：兩者各半
    alpha 接近 1：偏精確匹配（適合訂單號、規格）
    alpha 接近 0：偏語意（適合開放式問題）
    """
    results = []
    for doc in docs:
        bm25 = bm25_score(query, doc["keywords"])
        vec  = vector_score(query, doc["content"])
        hybrid = alpha * bm25 + (1 - alpha) * vec
        results.append({"doc": doc, "bm25": bm25, "vec": vec, "hybrid": hybrid})
    results.sort(key=lambda x: x["hybrid"], reverse=True)
    return results[:top_k]

# 展示：精確查詢 BM25 的優勢
queries = [
    ("退貨要幾天？",   0.5),   # 語意查詢
    ("7天退貨政策",   0.8),   # 精確查詢（數字）
    ("如何付款",       0.3),   # 語意查詢
]

print("混合搜尋 vs 純向量搜尋比較：")
for query, alpha in queries:
    results = hybrid_search(query, KB_DOCUMENTS, alpha=alpha)
    vec_only = hybrid_search(query, KB_DOCUMENTS, alpha=0.0)  # 純向量
    print(f"\n查詢：'{query}' (alpha={alpha})")
    print(f"  混合搜尋 Top 1: {results[0]['doc']['id']} → {results[0]['doc']['content'][:50]}")
    print(f"  純向量 Top 1:   {vec_only[0]['doc']['id']} → {vec_only[0]['doc']['content'][:50]}")

## 決策 3：升級人工的判斷 — 為什麼不純靠 LLM 決定？

In [ ]:
class EscalationJudge:
    """
    雙層升級判斷：Rule（快）+ LLM（準）
    
    為什麼不純靠 LLM？
    - LLM 可能「覺得自己能處理」，導致不必要的風險
    - 投訴、法律問題要100%升人工，不能靠 LLM 判斷
    - Rule 成本接近零，LLM 有成本
    
    為什麼不純靠 Rule？
    - Rule 無法覆蓋所有情況（語言多樣性）
    - 複雜的情感分析靠規則做得不好
    """
    
    # Layer 1：Rule-based（零成本，100% 確定性）
    MANDATORY_ESCALATION_RULES = [
        (lambda intent, _: intent == Intent.COMPLAINT,            "投訴一律升人工"),
        (lambda _, text: "律師" in text or "法律" in text or "告你" in text, "法律威脅"),
        (lambda _, text: "媒體" in text or "爆料" in text,         "媒體威脅"),
        (lambda _, text: text.count("！") >= 3,                   "強烈情緒"),
        (lambda _, text: "死" in text or "傷" in text,            "人身安全"),
    ]
    
    def check_rules(self, intent: Intent, user_text: str) -> tuple[bool, str]:
        """Layer 1：規則過濾（< 1ms）"""
        for rule_fn, reason in self.MANDATORY_ESCALATION_RULES:
            if rule_fn(intent, user_text):
                return True, reason
        return False, ""
    
    async def check_llm(self, conversation_history: list[str], ai_response: str) -> tuple[bool, str]:
        """Layer 2：LLM 判斷（用於複雜情緒和情境）"""
        context = "\n".join(conversation_history[-4:])
        prompt = f"""評估以下客服對話，判斷是否需要升級人工客服。
        
對話：
{context}

AI 回答：{ai_response[:200]}

評估標準：
- AI 回答不確定或包含「建議諮詢」等詞
- 用戶重複問了 3 次以上
- 問題超出 AI 能力範圍
- 用戶明顯不滿意

只回覆：ESCALATE 或 AI_HANDLE，加上原因。格式：ESCALATE:原因 或 AI_HANDLE:原因"""
        
        if LLM_AVAILABLE:
            resp = await llm_fast.ainvoke([HumanMessage(content=prompt)])
            result = resp.content.strip()
        else:
            # 模擬：如果 AI 回應包含「建議」就升人工
            result = "ESCALATE:AI 回答不確定" if "建議" in ai_response else "AI_HANDLE:正常處理"
        
        escalate = result.startswith("ESCALATE")
        reason = result.split(":", 1)[1] if ":" in result else result
        return escalate, reason
    
    async def should_escalate(self, intent: Intent, user_text: str,
                               history: list[str], ai_response: str) -> dict:
        # Layer 1（幾乎不花時間）
        rule_escalate, rule_reason = self.check_rules(intent, user_text)
        if rule_escalate:
            return {"escalate": True, "layer": "rule", "reason": rule_reason}
        
        # Layer 2（只在 Layer 1 通過時才呼叫）
        llm_escalate, llm_reason = await self.check_llm(history, ai_response)
        return {"escalate": llm_escalate, "layer": "llm", "reason": llm_reason}


judge = EscalationJudge()

test_cases = [
    (Intent.ORDER_QUERY, "我的訂單到哪了",      "您的訂單預計明天到達。"),
    (Intent.COMPLAINT,   "你們服務太差！！！！", "非常抱歉..."),
    (Intent.GENERAL_FAQ, "我要找律師告你們！",  "您好，關於您的問題..."),
    (Intent.TECH_SUPPORT, "APP 閃退怎麼辦",     "建議您先重新安裝 APP，如果仍有問題請聯繫技術支援。"),
]

print("升級判斷測試：")
for intent, user_text, ai_resp in test_cases:
    result = await judge.should_escalate(intent, user_text, [user_text], ai_resp)
    icon = "🆙 升人工" if result["escalate"] else "🤖 AI處理"
    print(f"  {icon} [{result['layer']}] '{user_text[:30]}' → {result['reason'][:40]}")

## 決策 4：Vertex AI Vector Search vs Pinecone

In [ ]:
print("""
向量資料庫選型決策：
═══════════════════════════════════════════════════════

選項比較（以台灣電商情境）：

                      Pinecone         Vertex AI Vector Search
───────────────────────────────────────────────────────
資料主權              ❌ 美國伺服器       ✅ 可選 asia-east1（台灣）
GCP 整合              需要 API 設定      ✅ 原生整合（同帳單、IAM）
維護成本              ✅ SaaS，零 ops     ✅ 類似，GCP 管理
定價（1M 向量）        ~$70/月           ~$50/月
延遲（asia）           ~80ms            ✅ ~20ms（同 region）
Metadata 過濾         ✅ 支援            ✅ 支援
CMEK                 ❌ 無              ✅ 支援（合規要求）
VPC-SC               ❌ 無              ✅ 支援（金融合規）
最大向量數            無上限             10B+

決策結果：選 Vertex AI Vector Search

原因：
1. 資料主權：客戶資料（訂單、對話）不出台灣
2. GCP 整合：已在 GCP 上，同帳單更好管理
3. 延遲：同 Region 的延遲顯著更低
4. 成本：稍便宜，且包含在 GCP 信用額度內

什麼情況選 Pinecone？
- 不在 GCP 環境（已在 AWS/Azure 上）
- 需要最快的 POC 速度（Pinecone 5 分鐘設好）
- 向量量很小，不在乎 $20/月 的差距
═══════════════════════════════════════════════════════
""")

## 決策 5：Context 管理 — 為什麼用 Summary Buffer 不用全歷史？

In [ ]:
class CustomerServiceContextManager:
    """
    客服場景的 Context 管理策略：
    Summary Buffer + Structured State
    
    為什麼不存全部歷史？
    - 客服對話平均 8-12 輪，累積 ~3,000 tokens
    - 加上 System Prompt + RAG context，容易超 10K
    - 每次請求都用這麼多 token，成本和延遲都不划算
    
    為什麼不用 Sliding Window？
    - 客服最重要的資訊常在第 1 輪（用戶說了訂單號碼）
    - Sliding 可能丟掉最關鍵的「問題描述」
    
    解法：Summary Buffer + Structured State
    - 結構化記錄關鍵資訊（order_id, issue_type）
    - 近期 3 輪完整保留
    - 舊的對話壓縮成 Case Summary
    """
    
    def __init__(self):
        # Structured State（永遠不丟失的關鍵資訊）
        self.case_state = {
            "order_id": None,
            "issue_type": None,
            "customer_name": None,
            "resolution_status": "open",
        }
        self.case_summary = ""   # 壓縮的歷史摘要
        self.recent_turns: list[dict] = []  # 最近 3 輪
        self.total_turns = 0
    
    def add_turn(self, role: str, content: str):
        self.total_turns += 1
        self.recent_turns.append({"role": role, "content": content})
        
        # 自動提取關鍵資訊更新 Structured State
        if role == "user":
            order_match = re.search(r'#(\d{4,})', content)
            if order_match:
                self.case_state["order_id"] = f"#{order_match.group(1)}"
            if "退貨" in content or "換貨" in content:
                self.case_state["issue_type"] = "return_refund"
        
        # 超過 3 輪：壓縮最舊的
        if len(self.recent_turns) > 6:  # 3 輪 × 2
            to_compress = self.recent_turns[:2]
            self.recent_turns = self.recent_turns[2:]
            compressed = " | ".join(f"{m['role']}: {m['content'][:40]}" for m in to_compress)
            self.case_summary = (self.case_summary + " → " + compressed).strip(" → ")
    
    def build_context_str(self) -> str:
        """組裝給 LLM 的 Context"""
        state_str = " | ".join(
            f"{k}={v}" for k, v in self.case_state.items() if v
        )
        parts = []
        if state_str:
            parts.append(f"[Case State] {state_str}")
        if self.case_summary:
            parts.append(f"[Summary] {self.case_summary}")
        parts.append("[Recent Turns]")
        for t in self.recent_turns:
            parts.append(f"  {t['role']}: {t['content']}")
        return "\n".join(parts)
    
    def token_estimate(self) -> int:
        return len(self.build_context_str().split()) * 4 // 3


ctx = CustomerServiceContextManager()

dialogue = [
    ("user", "我的訂單 #1234 在哪裡？"),
    ("assistant", "您的訂單 #1234 目前在新北配送中心，預計明天到達。"),
    ("user", "好的，那我要退貨怎麼辦？"),
    ("assistant", "退貨需在 7 天內申請，請點選 APP 中的退貨申請。"),
    ("user", "我的 APP 登不進去"),
    ("assistant", "請問您看到的錯誤訊息是什麼？"),
    ("user", "顯示密碼錯誤"),
    ("assistant", "請試試忘記密碼功能重設。"),
]

print("Context 管理演示：")
print("=" * 55)
for role, content in dialogue:
    ctx.add_turn(role, content)

print(ctx.build_context_str())
print(f"\nContext 大小：{ctx.token_estimate()} tokens（估算）")
print(f"如果全存：~{len(dialogue) * 50} tokens")
print(f"節省：~{(1 - ctx.token_estimate() / (len(dialogue) * 50)) * 100:.0f}%")
print(f"\n關鍵資訊（Structured State）永遠不丟失：{ctx.case_state}")

## 架構決策速查 + 面試白板語言

In [ ]:
print("""
FDE 面試：客服 Agent 系統設計的核心決策
═══════════════════════════════════════════════════════

Q: 為什麼用 Intent Router 而不是一個大 LLM？
A: 不同 Intent 的處理需求完全不同。
   訂單查詢只需要工具呼叫（無需 RAG），可以用 Flash；
   退換貨需要政策理解（需要 RAG）+ 同理心，需要 Pro。
   單一 LLM 的 System Prompt 會越來越複雜，品質下降，
   而且所有查詢都付 Pro 的錢，成本比路由方案高 3-5 倍。

Q: 為什麼用混合搜尋不用純向量？
A: 客服場景有很多精確查詢：訂單號碼、退貨天數、金額。
   純向量搜尋在語意上找不到「7天」或「#1234」。
   BM25 處理精確匹配，向量處理語意理解，兩者互補。
   我們用 alpha=0.7（偏精確）處理訂單查詢，
   alpha=0.3（偏語意）處理 FAQ。

Q: 升級人工的判斷為什麼要兩層？
A: Rule 層處理明確風險（投訴、法律威脅），零成本、100% 確定。
   LLM 層處理複雜情緒和情境，比 Rule 更準確但有成本。
   只有通過 Rule 層的才呼叫 LLM 判斷，
   大概 20% 的查詢需要 LLM 判斷，80% 在 Rule 層就決定了。

Q: 為什麼選 Vertex AI Vector Search 不選 Pinecone？
A: 三個核心原因：
   1. 資料主權：電商的訂單資料不能出台灣，GCP asia-east1 符合
   2. 整合：已在 GCP，同帳單、IAM、VPC 一套管
   3. 延遲：同 Region 的 PSC 路徑比跨國 Pinecone 快 4-5 倍
   唯一選 Pinecone 的情況：POC 階段，5 分鐘就能試起來。
═══════════════════════════════════════════════════════
""")